# Deep Learning Applications in Computer Vision Workshop

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/njpinton/CMSC178IP/blob/main/10-ComputerVisionDeepLearningII/notebooks/deep_learning_applications_workshop.ipynb)

---

## Workshop Objectives

By the end of this 60-minute workshop, you will:

1. Implement advanced image classification on CIFAR-10 and MNIST datasets
2. Understand and apply semantic segmentation using U-Net architecture
3. Explore object detection concepts with YOLO and Faster R-CNN
4. Build practical computer vision applications using deep learning
5. Apply transfer learning and fine-tuning techniques
6. Visualize and interpret model predictions across different tasks

**Duration:** 45-60 minutes

**Prerequisites:** Basic deep learning knowledge, CNNs, Python programming

---

## Setup & Imports

Let's prepare our environment with all necessary libraries.

In [ ]:
# Core libraries
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import cv2
from PIL import Image
import os

# Deep learning framework
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import cifar10, mnist
from tensorflow.keras.applications import VGG16, ResNet50
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Utilities
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Configure matplotlib
plt.style.use('default')
%matplotlib inline

print(f"TensorFlow version: {tf.__version__}")
print(f"OpenCV version: {cv2.__version__}")
print("Setup complete!")

---

## Part 1: Advanced Image Classification

### 1.1 Multi-Dataset Classification

We'll work with two classic datasets:
- **MNIST**: Handwritten digits (28×28 grayscale)
- **CIFAR-10**: Natural images (32×32 color)

This demonstrates how different datasets require different architectural considerations.

In [ ]:
# Load datasets
(X_mnist_train, y_mnist_train), (X_mnist_test, y_mnist_test) = mnist.load_data()
(X_cifar_train, y_cifar_train), (X_cifar_test, y_cifar_test) = cifar10.load_data()

# CIFAR-10 class names
cifar_classes = ['airplane', 'automobile', 'bird', 'cat', 'deer',
                 'dog', 'frog', 'horse', 'ship', 'truck']

# Use subsets for faster training
X_mnist_train = X_mnist_train[:5000]
y_mnist_train = y_mnist_train[:5000]
X_mnist_test = X_mnist_test[:1000]
y_mnist_test = y_mnist_test[:1000]

X_cifar_train = X_cifar_train[:5000]
y_cifar_train = y_cifar_train[:5000]
X_cifar_test = X_cifar_test[:1000]
y_cifar_test = y_cifar_test[:1000]

print(f"MNIST - Train: {X_mnist_train.shape}, Test: {X_mnist_test.shape}")
print(f"CIFAR-10 - Train: {X_cifar_train.shape}, Test: {X_cifar_test.shape}")

In [ ]:
# Visualize both datasets
fig, axes = plt.subplots(2, 10, figsize=(15, 4))
fig.suptitle('Dataset Comparison: MNIST vs CIFAR-10', fontsize=14, fontweight='bold')

# MNIST samples
for i in range(10):
    axes[0, i].imshow(X_mnist_train[i], cmap='gray')
    axes[0, i].set_title(f"{y_mnist_train[i]}", fontsize=9)
    axes[0, i].axis('off')
    if i == 0:
        axes[0, i].set_ylabel('MNIST', fontsize=12, fontweight='bold')

# CIFAR-10 samples
for i in range(10):
    axes[1, i].imshow(X_cifar_train[i])
    axes[1, i].set_title(f"{cifar_classes[y_cifar_train[i][0]]}", fontsize=8)
    axes[1, i].axis('off')
    if i == 0:
        axes[1, i].set_ylabel('CIFAR-10', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

### 1.2 Data Preprocessing

Different datasets require different preprocessing strategies.

In [ ]:
# Preprocess MNIST
X_mnist_train = X_mnist_train.reshape(-1, 28, 28, 1).astype('float32') / 255.0
X_mnist_test = X_mnist_test.reshape(-1, 28, 28, 1).astype('float32') / 255.0
y_mnist_train_cat = to_categorical(y_mnist_train, 10)
y_mnist_test_cat = to_categorical(y_mnist_test, 10)

# Preprocess CIFAR-10
X_cifar_train = X_cifar_train.astype('float32') / 255.0
X_cifar_test = X_cifar_test.astype('float32') / 255.0
y_cifar_train_cat = to_categorical(y_cifar_train, 10)
y_cifar_test_cat = to_categorical(y_cifar_test, 10)

print("Preprocessed shapes:")
print(f"MNIST: {X_mnist_train.shape}")
print(f"CIFAR-10: {X_cifar_train.shape}")

### 1.3 Building Specialized CNNs

We'll create architectures optimized for each dataset type.

In [ ]:
def create_mnist_cnn():
    """CNN optimized for grayscale digit classification."""
    model = models.Sequential([
        # Input: 28x28x1
        layers.Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=(28, 28, 1)),
        layers.BatchNormalization(),
        layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),
        
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),
        
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        layers.Dense(10, activation='softmax')
    ], name='MNIST_CNN')
    return model

def create_cifar_cnn():
    """CNN optimized for color natural image classification."""
    model = models.Sequential([
        # Input: 32x32x3
        layers.Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=(32, 32, 3)),
        layers.BatchNormalization(),
        layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.2),
        
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.3),
        
        layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.4),
        
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        layers.Dense(10, activation='softmax')
    ], name='CIFAR_CNN')
    return model

# Create models
mnist_model = create_mnist_cnn()
cifar_model = create_cifar_cnn()

# Compile
mnist_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
cifar_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

print("\nMNIST Model:")
mnist_model.summary()
print("\nCIFAR-10 Model:")
cifar_model.summary()

### 1.4 Training with Data Augmentation

In [ ]:
# Data augmentation for CIFAR-10
cifar_datagen = ImageDataGenerator(
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    zoom_range=0.1
)

# Train MNIST model
print("Training MNIST model...")
mnist_history = mnist_model.fit(
    X_mnist_train, y_mnist_train_cat,
    epochs=8,
    batch_size=128,
    validation_split=0.1,
    verbose=1
)

# Train CIFAR-10 model with augmentation
print("\nTraining CIFAR-10 model with data augmentation...")
cifar_history = cifar_model.fit(
    cifar_datagen.flow(X_cifar_train, y_cifar_train_cat, batch_size=64),
    steps_per_epoch=len(X_cifar_train) // 64,
    epochs=10,
    validation_data=(X_cifar_test, y_cifar_test_cat),
    verbose=1
)

In [ ]:
# Visualize training history
def plot_dual_history(history1, history2, name1, name2):
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # MNIST loss
    axes[0, 0].plot(history1.history['loss'], 'b-', label='Train', linewidth=2)
    axes[0, 0].plot(history1.history['val_loss'], 'r-', label='Val', linewidth=2)
    axes[0, 0].set_title(f'{name1} - Loss', fontsize=12, fontweight='bold')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # MNIST accuracy
    axes[0, 1].plot(history1.history['accuracy'], 'b-', label='Train', linewidth=2)
    axes[0, 1].plot(history1.history['val_accuracy'], 'r-', label='Val', linewidth=2)
    axes[0, 1].set_title(f'{name1} - Accuracy', fontsize=12, fontweight='bold')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('Accuracy')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    
    # CIFAR-10 loss
    axes[1, 0].plot(history2.history['loss'], 'b-', label='Train', linewidth=2)
    axes[1, 0].plot(history2.history['val_loss'], 'r-', label='Val', linewidth=2)
    axes[1, 0].set_title(f'{name2} - Loss', fontsize=12, fontweight='bold')
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('Loss')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    
    # CIFAR-10 accuracy
    axes[1, 1].plot(history2.history['accuracy'], 'b-', label='Train', linewidth=2)
    axes[1, 1].plot(history2.history['val_accuracy'], 'r-', label='Val', linewidth=2)
    axes[1, 1].set_title(f'{name2} - Accuracy', fontsize=12, fontweight='bold')
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].set_ylabel('Accuracy')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

plot_dual_history(mnist_history, cifar_history, 'MNIST', 'CIFAR-10')

### 1.5 Model Evaluation and Comparison

In [ ]:
# Evaluate both models
mnist_loss, mnist_acc = mnist_model.evaluate(X_mnist_test, y_mnist_test_cat, verbose=0)
cifar_loss, cifar_acc = cifar_model.evaluate(X_cifar_test, y_cifar_test_cat, verbose=0)

print(f"MNIST Test Accuracy: {mnist_acc*100:.2f}%")
print(f"CIFAR-10 Test Accuracy: {cifar_acc*100:.2f}%")

# Predictions
mnist_preds = np.argmax(mnist_model.predict(X_mnist_test, verbose=0), axis=1)
cifar_preds = np.argmax(cifar_model.predict(X_cifar_test, verbose=0), axis=1)

# Visualize sample predictions
fig, axes = plt.subplots(2, 8, figsize=(16, 5))
fig.suptitle('Classification Results Comparison', fontsize=14, fontweight='bold')

for i in range(8):
    # MNIST predictions
    axes[0, i].imshow(X_mnist_test[i].reshape(28, 28), cmap='gray')
    true_label = y_mnist_test[i]
    pred_label = mnist_preds[i]
    color = 'green' if true_label == pred_label else 'red'
    axes[0, i].set_title(f"T:{true_label}\nP:{pred_label}", color=color, fontsize=9)
    axes[0, i].axis('off')
    if i == 0:
        axes[0, i].set_ylabel('MNIST', fontsize=11, fontweight='bold')
    
    # CIFAR-10 predictions
    axes[1, i].imshow(X_cifar_test[i])
    true_label = y_cifar_test[i][0]
    pred_label = cifar_preds[i]
    color = 'green' if true_label == pred_label else 'red'
    axes[1, i].set_title(f"{cifar_classes[pred_label]}", color=color, fontsize=8)
    axes[1, i].axis('off')
    if i == 0:
        axes[1, i].set_ylabel('CIFAR-10', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

---

## Part 2: Image Segmentation with U-Net

### 2.1 Understanding Semantic Segmentation

**Semantic segmentation** assigns a class label to every pixel in an image.

**U-Net Architecture:**
- **Encoder**: Downsampling path (like a CNN)
- **Bottleneck**: Lowest resolution, highest level features
- **Decoder**: Upsampling path with skip connections
- **Output**: Pixel-wise classification map

We'll create a simple synthetic segmentation task for demonstration.

In [ ]:
def generate_synthetic_images(n_samples=500, img_size=128):
    """
    Generate synthetic images with circles and rectangles for segmentation.
    Returns images and corresponding segmentation masks.
    """
    images = []
    masks = []
    
    for _ in range(n_samples):
        # Create blank image and mask
        img = np.zeros((img_size, img_size, 3), dtype=np.uint8)
        mask = np.zeros((img_size, img_size), dtype=np.uint8)
        
        # Randomly add shapes
        n_shapes = np.random.randint(2, 5)
        
        for _ in range(n_shapes):
            shape_type = np.random.choice(['circle', 'rectangle'])
            
            if shape_type == 'circle':
                center = (np.random.randint(20, img_size-20), np.random.randint(20, img_size-20))
                radius = np.random.randint(10, 30)
                color = tuple(np.random.randint(100, 255, 3).tolist())
                cv2.circle(img, center, radius, color, -1)
                cv2.circle(mask, center, radius, 1, -1)  # Class 1: circle
            else:
                pt1 = (np.random.randint(10, img_size-40), np.random.randint(10, img_size-40))
                pt2 = (pt1[0] + np.random.randint(20, 40), pt1[1] + np.random.randint(20, 40))
                color = tuple(np.random.randint(100, 255, 3).tolist())
                cv2.rectangle(img, pt1, pt2, color, -1)
                cv2.rectangle(mask, pt1, pt2, 2, -1)  # Class 2: rectangle
        
        images.append(img)
        masks.append(mask)
    
    return np.array(images), np.array(masks)

# Generate dataset
print("Generating synthetic segmentation dataset...")
seg_images, seg_masks = generate_synthetic_images(n_samples=500, img_size=128)

# Normalize images
seg_images = seg_images.astype('float32') / 255.0

# Split dataset
X_seg_train, X_seg_test, y_seg_train, y_seg_test = train_test_split(
    seg_images, seg_masks, test_size=0.2, random_state=42
)

print(f"Segmentation dataset - Train: {X_seg_train.shape}, Test: {X_seg_test.shape}")

In [ ]:
# Visualize synthetic data
fig, axes = plt.subplots(3, 6, figsize=(15, 8))
fig.suptitle('Synthetic Segmentation Dataset', fontsize=14, fontweight='bold')

for i in range(6):
    # Original image
    axes[0, i].imshow(X_seg_train[i])
    axes[0, i].axis('off')
    if i == 0:
        axes[0, i].set_ylabel('Image', fontsize=11, fontweight='bold')
    
    # Ground truth mask
    axes[1, i].imshow(y_seg_train[i], cmap='tab10', vmin=0, vmax=2)
    axes[1, i].axis('off')
    if i == 0:
        axes[1, i].set_ylabel('Ground Truth', fontsize=11, fontweight='bold')
    
    # Overlay
    overlay = X_seg_train[i].copy()
    mask_colored = np.zeros_like(overlay)
    mask_colored[y_seg_train[i] == 1] = [0, 1, 0]  # Green for circles
    mask_colored[y_seg_train[i] == 2] = [1, 0, 0]  # Red for rectangles
    overlay = overlay * 0.6 + mask_colored * 0.4
    axes[2, i].imshow(overlay)
    axes[2, i].axis('off')
    if i == 0:
        axes[2, i].set_ylabel('Overlay', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

### 2.2 Building U-Net Architecture

U-Net consists of:
1. **Contracting path** (encoder): Captures context
2. **Bottleneck**: Connects encoder and decoder
3. **Expanding path** (decoder): Enables precise localization
4. **Skip connections**: Combine low and high-level features

In [ ]:
def create_unet(input_shape=(128, 128, 3), num_classes=3):
    """
    Build a U-Net model for semantic segmentation.
    
    Args:
        input_shape: Input image dimensions
        num_classes: Number of segmentation classes (including background)
    
    Returns:
        Keras model
    """
    inputs = layers.Input(shape=input_shape)
    
    # Encoder (Contracting path)
    c1 = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(inputs)
    c1 = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(c1)
    p1 = layers.MaxPooling2D((2, 2))(c1)
    
    c2 = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(p1)
    c2 = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(c2)
    p2 = layers.MaxPooling2D((2, 2))(c2)
    
    c3 = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(p2)
    c3 = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(c3)
    p3 = layers.MaxPooling2D((2, 2))(c3)
    
    # Bottleneck
    c4 = layers.Conv2D(256, (3, 3), activation='relu', padding='same')(p3)
    c4 = layers.Conv2D(256, (3, 3), activation='relu', padding='same')(c4)
    
    # Decoder (Expanding path)
    u5 = layers.Conv2DTranspose(128, (2, 2), strides=(2, 2), padding='same')(c4)
    u5 = layers.concatenate([u5, c3])  # Skip connection
    c5 = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(u5)
    c5 = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(c5)
    
    u6 = layers.Conv2DTranspose(64, (2, 2), strides=(2, 2), padding='same')(c5)
    u6 = layers.concatenate([u6, c2])  # Skip connection
    c6 = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(u6)
    c6 = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(c6)
    
    u7 = layers.Conv2DTranspose(32, (2, 2), strides=(2, 2), padding='same')(c6)
    u7 = layers.concatenate([u7, c1])  # Skip connection
    c7 = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(u7)
    c7 = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(c7)
    
    # Output layer
    outputs = layers.Conv2D(num_classes, (1, 1), activation='softmax')(c7)
    
    model = models.Model(inputs=inputs, outputs=outputs, name='U-Net')
    return model

# Create U-Net model
unet_model = create_unet(input_shape=(128, 128, 3), num_classes=3)
unet_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

unet_model.summary()

### 2.3 Training U-Net

In [ ]:
# Expand mask dimensions for training
y_seg_train_expanded = y_seg_train[..., np.newaxis]
y_seg_test_expanded = y_seg_test[..., np.newaxis]

# Train U-Net
print("Training U-Net for semantic segmentation...")
unet_history = unet_model.fit(
    X_seg_train, y_seg_train,
    epochs=15,
    batch_size=16,
    validation_split=0.15,
    verbose=1
)

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(unet_history.history['loss'], 'b-', label='Training', linewidth=2)
axes[0].plot(unet_history.history['val_loss'], 'r-', label='Validation', linewidth=2)
axes[0].set_title('U-Net Training Loss', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(unet_history.history['accuracy'], 'b-', label='Training', linewidth=2)
axes[1].plot(unet_history.history['val_accuracy'], 'r-', label='Validation', linewidth=2)
axes[1].set_title('U-Net Training Accuracy', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 2.4 Segmentation Results Visualization

In [ ]:
# Generate predictions
seg_predictions = unet_model.predict(X_seg_test, verbose=0)
seg_pred_masks = np.argmax(seg_predictions, axis=-1)

# Visualize segmentation results
fig, axes = plt.subplots(4, 6, figsize=(16, 11))
fig.suptitle('U-Net Segmentation Results', fontsize=14, fontweight='bold')

for i in range(6):
    # Original image
    axes[0, i].imshow(X_seg_test[i])
    axes[0, i].axis('off')
    if i == 0:
        axes[0, i].set_ylabel('Input Image', fontsize=10, fontweight='bold')
    
    # Ground truth mask
    axes[1, i].imshow(y_seg_test[i], cmap='tab10', vmin=0, vmax=2)
    axes[1, i].axis('off')
    if i == 0:
        axes[1, i].set_ylabel('Ground Truth', fontsize=10, fontweight='bold')
    
    # Predicted mask
    axes[2, i].imshow(seg_pred_masks[i], cmap='tab10', vmin=0, vmax=2)
    axes[2, i].axis('off')
    if i == 0:
        axes[2, i].set_ylabel('Prediction', fontsize=10, fontweight='bold')
    
    # Overlay prediction on image
    overlay = X_seg_test[i].copy()
    mask_colored = np.zeros_like(overlay)
    mask_colored[seg_pred_masks[i] == 1] = [0, 1, 0]  # Green for circles
    mask_colored[seg_pred_masks[i] == 2] = [1, 0, 0]  # Red for rectangles
    overlay = overlay * 0.6 + mask_colored * 0.4
    axes[3, i].imshow(overlay)
    axes[3, i].axis('off')
    if i == 0:
        axes[3, i].set_ylabel('Overlay', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

# Calculate IoU (Intersection over Union)
def calculate_iou(y_true, y_pred, class_id):
    intersection = np.logical_and(y_true == class_id, y_pred == class_id).sum()
    union = np.logical_or(y_true == class_id, y_pred == class_id).sum()
    return intersection / (union + 1e-7)

iou_bg = calculate_iou(y_seg_test, seg_pred_masks, 0)
iou_circle = calculate_iou(y_seg_test, seg_pred_masks, 1)
iou_rect = calculate_iou(y_seg_test, seg_pred_masks, 2)
mean_iou = (iou_bg + iou_circle + iou_rect) / 3

print(f"\nIoU Scores:")
print(f"Background: {iou_bg:.4f}")
print(f"Circles: {iou_circle:.4f}")
print(f"Rectangles: {iou_rect:.4f}")
print(f"Mean IoU: {mean_iou:.4f}")

---

## Part 3: Object Detection Concepts

### 3.1 Understanding Object Detection

**Object Detection** = **Classification** + **Localization**

**Key Components:**
1. **Bounding Box Regression**: Predict (x, y, width, height)
2. **Class Prediction**: What object is in the box?
3. **Confidence Score**: How certain is the prediction?

**Popular Architectures:**
- **YOLO (You Only Look Once)**: Single-stage, very fast
- **Faster R-CNN**: Two-stage, more accurate
- **SSD (Single Shot Detector)**: Balance of speed and accuracy

### 3.2 YOLO Overview

**YOLO Algorithm:**
1. Divide image into SxS grid
2. Each cell predicts B bounding boxes
3. Each box predicts: (x, y, w, h, confidence, class_probabilities)
4. Apply Non-Maximum Suppression (NMS) to filter boxes

In [ ]:
# Visualize YOLO concept
def visualize_yolo_concept():
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    # Create sample image
    img = np.ones((224, 224, 3))
    
    # Grid overlay
    grid_size = 7
    cell_size = 224 // grid_size
    
    axes[0].imshow(img)
    axes[0].set_title('1. Input Image', fontsize=12, fontweight='bold')
    for i in range(grid_size + 1):
        axes[0].axhline(y=i*cell_size, color='red', linewidth=1)
        axes[0].axvline(x=i*cell_size, color='red', linewidth=1)
    axes[0].axis('off')
    
    # Bounding boxes
    axes[1].imshow(img)
    axes[1].set_title('2. Predicted Bounding Boxes', fontsize=12, fontweight='bold')
    boxes = [
        (40, 50, 80, 100, 0.9),
        (120, 80, 70, 90, 0.85),
        (150, 140, 60, 50, 0.75)
    ]
    for x, y, w, h, conf in boxes:
        rect = Rectangle((x, y), w, h, linewidth=2, edgecolor='lime', facecolor='none')
        axes[1].add_patch(rect)
        axes[1].text(x, y-5, f'{conf:.2f}', color='lime', fontsize=10, fontweight='bold')
    axes[1].axis('off')
    
    # After NMS
    axes[2].imshow(img)
    axes[2].set_title('3. After Non-Maximum Suppression', fontsize=12, fontweight='bold')
    final_boxes = [(40, 50, 80, 100, 0.9, 'car'), (120, 80, 70, 90, 0.85, 'person')]
    for x, y, w, h, conf, label in final_boxes:
        rect = Rectangle((x, y), w, h, linewidth=3, edgecolor='yellow', facecolor='none')
        axes[2].add_patch(rect)
        axes[2].text(x, y-5, f'{label} {conf:.2f}', color='yellow', 
                    fontsize=10, fontweight='bold', 
                    bbox=dict(boxstyle='round', facecolor='black', alpha=0.5))
    axes[2].axis('off')
    
    plt.tight_layout()
    plt.show()

visualize_yolo_concept()

### 3.3 Faster R-CNN Overview

**Two-Stage Architecture:**

**Stage 1: Region Proposal Network (RPN)**
- Scans feature maps
- Proposes potential object regions
- Outputs region of interest (RoI)

**Stage 2: Classification & Refinement**
- Extracts features from each RoI
- Classifies object
- Refines bounding box coordinates

**Advantages:**
- Higher accuracy than YOLO
- Better for small objects
- State-of-the-art performance

**Disadvantages:**
- Slower than YOLO
- More complex architecture
- Harder to train

In [ ]:
# Visualize architecture comparison
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# YOLO pipeline
axes[0].text(0.5, 0.9, 'YOLO (Single-Stage)', ha='center', fontsize=14, 
            fontweight='bold', transform=axes[0].transAxes)
pipeline_yolo = ['Input\nImage', 'CNN\nBackbone', 'Detection\nHead', 'Predictions\n(Boxes + Classes)']
x_pos = np.linspace(0.1, 0.9, len(pipeline_yolo))
for i, (x, stage) in enumerate(zip(x_pos, pipeline_yolo)):
    axes[0].add_patch(Rectangle((x-0.08, 0.3), 0.16, 0.4, 
                                facecolor='lightblue', edgecolor='black', linewidth=2))
    axes[0].text(x, 0.5, stage, ha='center', va='center', fontsize=10, fontweight='bold')
    if i < len(pipeline_yolo) - 1:
        axes[0].annotate('', xy=(x_pos[i+1]-0.08, 0.5), xytext=(x+0.08, 0.5),
                        arrowprops=dict(arrowstyle='->', lw=2, color='black'))
axes[0].set_xlim(0, 1)
axes[0].set_ylim(0, 1)
axes[0].axis('off')
axes[0].text(0.95, 0.05, 'Fast (~45 FPS)', ha='right', fontsize=11, 
            style='italic', transform=axes[0].transAxes, color='green')

# Faster R-CNN pipeline
axes[1].text(0.5, 0.9, 'Faster R-CNN (Two-Stage)', ha='center', fontsize=14,
            fontweight='bold', transform=axes[1].transAxes)
pipeline_rcnn = ['Input\nImage', 'CNN\nBackbone', 'RPN\n(Proposals)', 'RoI\nPooling', 
                'Classifier +\nBox Regressor', 'Predictions']
x_pos = np.linspace(0.05, 0.95, len(pipeline_rcnn))
for i, (x, stage) in enumerate(zip(x_pos, pipeline_rcnn)):
    color = 'lightcoral' if i in [2, 3] else 'lightblue'
    axes[1].add_patch(Rectangle((x-0.06, 0.3), 0.12, 0.4,
                                facecolor=color, edgecolor='black', linewidth=2))
    axes[1].text(x, 0.5, stage, ha='center', va='center', fontsize=9, fontweight='bold')
    if i < len(pipeline_rcnn) - 1:
        axes[1].annotate('', xy=(x_pos[i+1]-0.06, 0.5), xytext=(x+0.06, 0.5),
                        arrowprops=dict(arrowstyle='->', lw=2, color='black'))
axes[1].set_xlim(0, 1)
axes[1].set_ylim(0, 1)
axes[1].axis('off')
axes[1].text(0.95, 0.05, 'Slower (~7 FPS), More Accurate', ha='right', fontsize=11,
            style='italic', transform=axes[1].transAxes, color='red')

plt.tight_layout()
plt.show()

### 3.4 Key Metrics for Object Detection

**Intersection over Union (IoU):**
$$IoU = \frac{\text{Area of Overlap}}{\text{Area of Union}}$$

**Precision & Recall:**
- **Precision**: Of all detections, how many were correct?
- **Recall**: Of all ground truth objects, how many were detected?

**Average Precision (AP):**
- Area under Precision-Recall curve
- Computed at different IoU thresholds

**Mean Average Precision (mAP):**
- Average of AP across all classes
- Standard metric for comparing detectors

In [ ]:
# Visualize IoU calculation
def visualize_iou():
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    # Example boxes
    gt_box = [50, 50, 100, 100]  # x, y, width, height
    pred_boxes = [
        ([60, 60, 100, 100], 'Good'),
        ([80, 80, 100, 100], 'Medium'),
        ([120, 120, 100, 100], 'Poor')
    ]
    
    for idx, (pred_box, quality) in enumerate(pred_boxes):
        ax = axes[idx]
        
        # Draw ground truth box
        gt_rect = Rectangle((gt_box[0], gt_box[1]), gt_box[2], gt_box[3],
                           linewidth=3, edgecolor='green', facecolor='green', alpha=0.3)
        ax.add_patch(gt_rect)
        
        # Draw predicted box
        pred_rect = Rectangle((pred_box[0], pred_box[1]), pred_box[2], pred_box[3],
                             linewidth=3, edgecolor='red', facecolor='red', alpha=0.3)
        ax.add_patch(pred_rect)
        
        # Calculate IoU
        x1_inter = max(gt_box[0], pred_box[0])
        y1_inter = max(gt_box[1], pred_box[1])
        x2_inter = min(gt_box[0] + gt_box[2], pred_box[0] + pred_box[2])
        y2_inter = min(gt_box[1] + gt_box[3], pred_box[1] + pred_box[3])
        
        if x2_inter > x1_inter and y2_inter > y1_inter:
            intersection = (x2_inter - x1_inter) * (y2_inter - y1_inter)
        else:
            intersection = 0
        
        area_gt = gt_box[2] * gt_box[3]
        area_pred = pred_box[2] * pred_box[3]
        union = area_gt + area_pred - intersection
        iou = intersection / union if union > 0 else 0
        
        ax.set_xlim(0, 250)
        ax.set_ylim(0, 250)
        ax.set_aspect('equal')
        ax.invert_yaxis()
        ax.set_title(f'{quality} IoU: {iou:.2f}', fontsize=12, fontweight='bold')
        ax.legend(['Ground Truth', 'Prediction'], loc='upper right')
        ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

visualize_iou()

### 3.5 Comparison Table

| Feature | YOLO | Faster R-CNN | SSD |
|---------|------|--------------|-----|
| **Speed** | Very Fast (45+ FPS) | Slow (5-7 FPS) | Fast (20-30 FPS) |
| **Accuracy** | Good | Excellent | Good |
| **Small Objects** | Weak | Strong | Medium |
| **Architecture** | Single-stage | Two-stage | Single-stage |
| **Training** | Easier | Complex | Medium |
| **Use Case** | Real-time apps | High accuracy needed | Balanced |

**When to use:**
- **YOLO**: Real-time video processing, robotics, autonomous vehicles
- **Faster R-CNN**: Medical imaging, satellite imagery, high-stakes applications
- **SSD**: Mobile applications, moderate real-time requirements

---

## Part 4: Transfer Learning

### 4.1 Using Pre-trained Models

Transfer learning leverages models trained on large datasets (ImageNet) for new tasks.

**Benefits:**
- Less training data needed
- Faster convergence
- Better generalization
- State-of-the-art features

In [ ]:
# Build transfer learning model for CIFAR-10
def create_transfer_model(input_shape=(32, 32, 3), num_classes=10):
    # Load pre-trained VGG16 (without top layers)
    base_model = VGG16(
        weights='imagenet',
        include_top=False,
        input_shape=(32, 32, 3)
    )
    
    # Freeze base model layers
    base_model.trainable = False
    
    # Add custom classification head
    model = models.Sequential([
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation='softmax')
    ], name='Transfer_VGG16')
    
    return model

# Create and compile transfer learning model
transfer_model = create_transfer_model()
transfer_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("Transfer Learning Model Architecture:")
transfer_model.summary()

In [ ]:
# Train transfer learning model
print("Training transfer learning model...")
transfer_history = transfer_model.fit(
    X_cifar_train, y_cifar_train_cat,
    epochs=10,
    batch_size=64,
    validation_split=0.15,
    verbose=1
)

# Evaluate
transfer_loss, transfer_acc = transfer_model.evaluate(
    X_cifar_test, y_cifar_test_cat, verbose=0
)
print(f"\nTransfer Learning Test Accuracy: {transfer_acc*100:.2f}%")
print(f"Original CNN Test Accuracy: {cifar_acc*100:.2f}%")
print(f"Improvement: {(transfer_acc - cifar_acc)*100:.2f}%")

---

## Part 5: Student Activity (15 minutes)

### Challenge: Build Your Own Multi-Task Vision System

**Objective:** Create a complete vision pipeline that demonstrates understanding of classification, segmentation, and detection concepts.

**Tasks:**

1. **Improve MNIST Classifier (5 min)**
   - Add data augmentation
   - Achieve >97% test accuracy
   - Visualize misclassified examples

2. **Enhance U-Net Segmentation (5 min)**
   - Add batch normalization to decoder
   - Increase training epochs to 20
   - Calculate and report IoU for each class

3. **Design Object Detection Simulation (5 min)**
   - Create synthetic images with multiple objects
   - Implement basic bounding box prediction
   - Visualize detection results

**Bonus Challenges:**
- Implement Non-Maximum Suppression (NMS)
- Fine-tune a pre-trained model on CIFAR-10
- Create a confusion matrix for segmentation classes

### Starter Code

In [ ]:
# Task 1: Improve MNIST Classifier
# TODO: Add your data augmentation and model improvements here

# Hint: Use ImageDataGenerator with appropriate transformations
# mnist_augmenter = ImageDataGenerator(...)

# TODO: Build improved model
# improved_mnist_model = ...

# TODO: Train and evaluate
# ...

In [ ]:
# Task 2: Enhance U-Net Segmentation
# TODO: Modify create_unet() to add BatchNormalization layers

# def create_enhanced_unet(...):
#     # Add BatchNormalization after each Conv2D
#     ...

# TODO: Train for 20 epochs
# enhanced_unet = create_enhanced_unet()
# ...

In [ ]:
# Task 3: Object Detection Simulation
# TODO: Create function to generate images with bounding box labels

# def generate_detection_dataset(n_samples=100):
#     # Generate images with known object locations
#     # Return: images, bounding_boxes, class_labels
#     ...

# TODO: Build simple detection model
# detection_model = ...

# TODO: Visualize predictions
# ...

### Reflection Questions

1. What are the key differences between classification, segmentation, and detection?
2. When would you choose U-Net over a standard CNN?
3. What makes YOLO faster than Faster R-CNN?
4. How does transfer learning help with limited training data?
5. What metrics would you use to evaluate each type of model?

---

## Part 6: Solutions

### Solution: Improved MNIST Classifier

In [ ]:
# Solution 1: Improved MNIST with data augmentation
mnist_augmenter = ImageDataGenerator(
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1
)

def create_improved_mnist_cnn():
    model = models.Sequential([
        layers.Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=(28, 28, 1)),
        layers.BatchNormalization(),
        layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),
        
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),
        
        layers.Flatten(),
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        layers.Dense(10, activation='softmax')
    ])
    return model

improved_mnist = create_improved_mnist_cnn()
improved_mnist.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

print("Training improved MNIST model with data augmentation...")
improved_mnist_history = improved_mnist.fit(
    mnist_augmenter.flow(X_mnist_train, y_mnist_train_cat, batch_size=128),
    steps_per_epoch=len(X_mnist_train) // 128,
    epochs=15,
    validation_data=(X_mnist_test, y_mnist_test_cat),
    verbose=1
)

improved_loss, improved_acc = improved_mnist.evaluate(X_mnist_test, y_mnist_test_cat, verbose=0)
print(f"\nImproved MNIST Accuracy: {improved_acc*100:.2f}%")

In [ ]:
# Visualize misclassified MNIST examples
improved_preds = np.argmax(improved_mnist.predict(X_mnist_test, verbose=0), axis=1)
misclassified_idx = np.where(improved_preds != y_mnist_test)[0]

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
fig.suptitle('Misclassified MNIST Examples', fontsize=14, fontweight='bold')

for i, idx in enumerate(misclassified_idx[:10]):
    ax = axes[i // 5, i % 5]
    ax.imshow(X_mnist_test[idx].reshape(28, 28), cmap='gray')
    ax.set_title(f"True: {y_mnist_test[idx]}\nPred: {improved_preds[idx]}", 
                fontsize=9, color='red')
    ax.axis('off')

plt.tight_layout()
plt.show()

print(f"Total misclassified: {len(misclassified_idx)} out of {len(y_mnist_test)}")
print(f"Error rate: {len(misclassified_idx)/len(y_mnist_test)*100:.2f}%")

### Solution: Enhanced U-Net with Batch Normalization

In [ ]:
# Solution 2: Enhanced U-Net
def create_enhanced_unet(input_shape=(128, 128, 3), num_classes=3):
    inputs = layers.Input(shape=input_shape)
    
    # Encoder
    c1 = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(inputs)
    c1 = layers.BatchNormalization()(c1)
    c1 = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(c1)
    c1 = layers.BatchNormalization()(c1)
    p1 = layers.MaxPooling2D((2, 2))(c1)
    
    c2 = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(p1)
    c2 = layers.BatchNormalization()(c2)
    c2 = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(c2)
    c2 = layers.BatchNormalization()(c2)
    p2 = layers.MaxPooling2D((2, 2))(c2)
    
    c3 = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(p2)
    c3 = layers.BatchNormalization()(c3)
    c3 = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(c3)
    c3 = layers.BatchNormalization()(c3)
    p3 = layers.MaxPooling2D((2, 2))(c3)
    
    # Bottleneck
    c4 = layers.Conv2D(256, (3, 3), activation='relu', padding='same')(p3)
    c4 = layers.BatchNormalization()(c4)
    c4 = layers.Conv2D(256, (3, 3), activation='relu', padding='same')(c4)
    c4 = layers.BatchNormalization()(c4)
    
    # Decoder
    u5 = layers.Conv2DTranspose(128, (2, 2), strides=(2, 2), padding='same')(c4)
    u5 = layers.concatenate([u5, c3])
    c5 = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(u5)
    c5 = layers.BatchNormalization()(c5)
    c5 = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(c5)
    c5 = layers.BatchNormalization()(c5)
    
    u6 = layers.Conv2DTranspose(64, (2, 2), strides=(2, 2), padding='same')(c5)
    u6 = layers.concatenate([u6, c2])
    c6 = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(u6)
    c6 = layers.BatchNormalization()(c6)
    c6 = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(c6)
    c6 = layers.BatchNormalization()(c6)
    
    u7 = layers.Conv2DTranspose(32, (2, 2), strides=(2, 2), padding='same')(c6)
    u7 = layers.concatenate([u7, c1])
    c7 = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(u7)
    c7 = layers.BatchNormalization()(c7)
    c7 = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(c7)
    c7 = layers.BatchNormalization()(c7)
    
    outputs = layers.Conv2D(num_classes, (1, 1), activation='softmax')(c7)
    
    return models.Model(inputs=inputs, outputs=outputs, name='Enhanced_UNet')

enhanced_unet = create_enhanced_unet()
enhanced_unet.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

print("Training enhanced U-Net...")
enhanced_unet_history = enhanced_unet.fit(
    X_seg_train, y_seg_train,
    epochs=20,
    batch_size=16,
    validation_split=0.15,
    verbose=1
)

In [ ]:
# Evaluate enhanced U-Net
enhanced_seg_preds = enhanced_unet.predict(X_seg_test, verbose=0)
enhanced_seg_masks = np.argmax(enhanced_seg_preds, axis=-1)

# Calculate IoU for each class
iou_scores = {}
class_names = ['Background', 'Circles', 'Rectangles']

for class_id, class_name in enumerate(class_names):
    iou = calculate_iou(y_seg_test, enhanced_seg_masks, class_id)
    iou_scores[class_name] = iou

print("\nEnhanced U-Net IoU Scores:")
for class_name, iou in iou_scores.items():
    print(f"{class_name}: {iou:.4f}")
print(f"Mean IoU: {np.mean(list(iou_scores.values())):.4f}")

# Visualize improvement
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
fig.suptitle('Enhanced U-Net Results', fontsize=14, fontweight='bold')

for i in range(4):
    axes[0, i].imshow(X_seg_test[i])
    axes[0, i].axis('off')
    if i == 0:
        axes[0, i].set_ylabel('Input', fontsize=11, fontweight='bold')
    
    axes[1, i].imshow(enhanced_seg_masks[i], cmap='tab10', vmin=0, vmax=2)
    axes[1, i].axis('off')
    if i == 0:
        axes[1, i].set_ylabel('Prediction', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

### Solution: Object Detection Simulation

In [ ]:
# Solution 3: Object detection simulation
def generate_detection_dataset(n_samples=200, img_size=128):
    """
    Generate synthetic images with bounding box annotations.
    Each image contains 1-3 objects of different classes.
    """
    images = []
    boxes = []  # Format: [x, y, w, h, class_id]
    
    for _ in range(n_samples):
        img = np.ones((img_size, img_size, 3), dtype=np.uint8) * 255
        img_boxes = []
        
        n_objects = np.random.randint(1, 4)
        
        for _ in range(n_objects):
            obj_type = np.random.choice(['circle', 'square', 'triangle'])
            
            if obj_type == 'circle':
                radius = np.random.randint(10, 20)
                cx = np.random.randint(radius+5, img_size-radius-5)
                cy = np.random.randint(radius+5, img_size-radius-5)
                color = (255, 0, 0)  # Red
                cv2.circle(img, (cx, cy), radius, color, -1)
                # Bounding box
                img_boxes.append([cx-radius, cy-radius, 2*radius, 2*radius, 0])
                
            elif obj_type == 'square':
                size = np.random.randint(20, 40)
                x = np.random.randint(5, img_size-size-5)
                y = np.random.randint(5, img_size-size-5)
                color = (0, 255, 0)  # Green
                cv2.rectangle(img, (x, y), (x+size, y+size), color, -1)
                img_boxes.append([x, y, size, size, 1])
                
            else:  # triangle
                size = np.random.randint(20, 35)
                cx = np.random.randint(size, img_size-size)
                cy = np.random.randint(size, img_size-size)
                pts = np.array([
                    [cx, cy-size],
                    [cx-size, cy+size],
                    [cx+size, cy+size]
                ], np.int32)
                color = (0, 0, 255)  # Blue
                cv2.fillPoly(img, [pts], color)
                img_boxes.append([cx-size, cy-size, 2*size, 2*size, 2])
        
        images.append(img)
        boxes.append(img_boxes)
    
    return np.array(images), boxes

# Generate detection dataset
det_images, det_boxes = generate_detection_dataset(n_samples=200)
det_images_norm = det_images.astype('float32') / 255.0

print(f"Generated {len(det_images)} detection images")

In [ ]:
# Visualize detection dataset
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
fig.suptitle('Object Detection Dataset with Ground Truth Boxes', fontsize=14, fontweight='bold')

class_names_det = ['Circle', 'Square', 'Triangle']
colors = [(255, 0, 0), (0, 255, 0), (0, 0, 255)]

for i in range(8):
    ax = axes[i // 4, i % 4]
    img_display = det_images[i].copy()
    
    # Draw bounding boxes
    for box in det_boxes[i]:
        x, y, w, h, class_id = box
        color = colors[class_id]
        cv2.rectangle(img_display, (x, y), (x+w, y+h), color, 2)
        cv2.putText(img_display, class_names_det[class_id], (x, y-5),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.4, color, 1)
    
    ax.imshow(cv2.cvtColor(img_display, cv2.COLOR_BGR2RGB))
    ax.axis('off')

plt.tight_layout()
plt.show()

---

## Part 7: Summary & Key Takeaways

### What We Learned

**1. Image Classification**
- Built specialized CNNs for different image types (grayscale vs color)
- Applied data augmentation for better generalization
- Used transfer learning to leverage pre-trained models
- Achieved high accuracy with proper architecture design

**2. Semantic Segmentation**
- Understood U-Net encoder-decoder architecture
- Implemented skip connections for precise localization
- Learned pixel-wise classification techniques
- Evaluated using IoU metric

**3. Object Detection**
- Compared YOLO (speed) vs Faster R-CNN (accuracy)
- Understood single-stage vs two-stage detectors
- Learned key metrics: IoU, mAP, precision, recall
- Recognized trade-offs between speed and accuracy

**4. Transfer Learning**
- Leveraged pre-trained models for better performance
- Reduced training time and data requirements
- Fine-tuned models for specific tasks

### Architecture Comparison

| Task | Architecture | Key Feature | Best For |
|------|-------------|-------------|----------|
| **Classification** | CNN | Feature extraction + softmax | Single object per image |
| **Segmentation** | U-Net | Encoder-decoder + skip connections | Pixel-level labeling |
| **Detection** | YOLO/Faster R-CNN | Bounding box regression | Multiple objects |

### Best Practices Checklist

- [ ] Normalize input data appropriately
- [ ] Use data augmentation for small datasets
- [ ] Add batch normalization for training stability
- [ ] Include dropout to prevent overfitting
- [ ] Choose architecture based on task requirements
- [ ] Monitor both training and validation metrics
- [ ] Use appropriate evaluation metrics (accuracy, IoU, mAP)
- [ ] Consider transfer learning when possible
- [ ] Balance model complexity with dataset size
- [ ] Visualize predictions to understand failures

### Common Pitfalls to Avoid

1. **Wrong architecture for task**: Don't use classification CNN for segmentation
2. **Insufficient data augmentation**: Leads to overfitting on small datasets
3. **Improper normalization**: Different models expect different input ranges
4. **Ignoring class imbalance**: Use weighted losses for imbalanced datasets
5. **Not using transfer learning**: Missing opportunity for better performance
6. **Wrong evaluation metric**: IoU for segmentation, mAP for detection

### Next Steps

**To advance your skills:**

1. **Advanced Architectures**:
   - ResNet, DenseNet for classification
   - Mask R-CNN for instance segmentation
   - YOLOv5/v8 for real-time detection

2. **Real-World Applications**:
   - Medical image segmentation
   - Autonomous driving (detection + segmentation)
   - Face recognition and verification
   - Satellite imagery analysis

3. **Advanced Techniques**:
   - Attention mechanisms
   - Vision transformers
   - Multi-task learning
   - Self-supervised learning

4. **Deployment**:
   - Model optimization (quantization, pruning)
   - TensorFlow Lite for mobile
   - ONNX for cross-framework compatibility
   - Docker containerization

### Resources

- **Papers**:
  - [U-Net: Convolutional Networks for Biomedical Image Segmentation](https://arxiv.org/abs/1505.04597)
  - [You Only Look Once: Unified, Real-Time Object Detection](https://arxiv.org/abs/1506.02640)
  - [Faster R-CNN: Towards Real-Time Object Detection](https://arxiv.org/abs/1506.01497)

- **Courses**:
  - CS231n: Convolutional Neural Networks for Visual Recognition
  - Fast.ai Practical Deep Learning for Coders
  - Deep Learning Specialization (Coursera)

- **Frameworks & Tools**:
  - TensorFlow Object Detection API
  - Detectron2 (Facebook AI Research)
  - MMDetection (OpenMMLab)
  - Ultralytics YOLOv8

---

**Congratulations on completing the Deep Learning Applications Workshop!**

You now have practical experience with the three fundamental computer vision tasks and understand how to choose and apply the right deep learning approach for each problem.